In [68]:
# ======================================================================
# DUBLIN PLANNING DATA (ready to use)
# ======================================================================
# Input:  the national processed planning file
#         (dataSet/processed_dataset/planning_dataset/planning_2021_onwards.csv)
#
# Same steps as dublin_ber_data:
#   1. load the planning CSV
#   2. keep only Dublin and add the postal district
#   3. remove the columns that are the same thing as the targets
#   4. remove columns with more than 50% missing values
#   5. remove constant columns
#   6. remove id / admin columns
#   7. put the columns in a clear order
#
# Targets (Student 2):
#   Decision          -> granted or refused (classification, Linear SVM)
#   days_to_decision  -> how long the council took (regression, ElasticNet)
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [69]:
# Folder paths

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

planning_file = os.path.join(base, 'dataSet', 'processed_dataset', 'planning_dataset', 'planning_2021_onwards.csv')

print(planning_file)

C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet\processed_dataset\planning_dataset\planning_2021_onwards.csv


In [70]:
# Loading the planning file (one CSV, one row per application)

df_planning = pd.read_csv(planning_file, low_memory=False)

print('Rows:', df_planning.shape[0], ' Columns:', df_planning.shape[1])
df_planning.head()

Rows: 225141  Columns: 36


,OBJECTID,PlanningAuthority,ApplicationNumber,DevelopmentDescription,DevelopmentAddress,DevelopmentPostcode,ApplicationStatus,ApplicationType,Decision,LandUseCode,AreaofSite,NumResidentialUnits,OneOffHouse,FloorArea,ReceivedDate,WithdrawnDate,DecisionDate,DecisionDueDate,GrantDate,ExpiryDate,AppealRefNumber,AppealStatus,AppealDecision,AppealDecisionDate,AppealSubmittedDate,FIRequestDate,FIRecDate,LinkAppDetails,OneOffKPI,ETL_DATE,SiteId,word_count,county,year,quarter,days_to_decision
0,632,Carlow County Council,21393,Permission is sought for vehicular entrance in...,"Dublin Road , Pollerton Big , CarlowTown",NaN,APPLICATION FINALISED,PERMISSION,REFUSED,,NaN,0.0,,0.0,2021-10-15,NaN,2022-10-19,2022-10-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-12-07,2022-08-26,http://www.eplanning.ie/CarlowCC/AppFileRefDet...,No,2026-10-20,NaN,43,Carlow,2021,2021Q4,369.0
1,886,Carlow County Council,2445,alterations and additions to an existing singl...,"Ballyveal , Ballon , Co. Carlow",NaN,APPLICATION FINALISED,PERMISSION,CONDITIONAL,,NaN,1.0,,10.0,2024-10-17,NaN,2024-12-09,2024-12-11,2025-01-20,2030-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDet...,No,2026-10-20,NaN,49,Carlow,2024,2024Q4,53.0
2,887,Carlow County Council,2446,"construct a single-storey domestic dwelling, w...","Ballypierce , Kildavin , Co.Carlow",NaN,WITHDRAWN,PERMISSION,NaN,,NaN,0.0,,0.0,2024-10-29,2024-11-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDet...,No,2026-10-20,NaN,31,Carlow,2024,2024Q4,NaN
3,888,Carlow County Council,2447,the construction of a one and a half storey ex...,"Rickettstown , Rathvilly , Co. Carlow",NaN,INCOMPLETED APPLICATION,PERMISSION,NaN,NaN,NaN,0.0,NaN,0.0,2024-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDet...,No,2026-10-20,NaN,33,Carlow,2024,2024Q4,NaN
4,889,Carlow County Council,2448,the erection of two houses and two garages,"Burrin Road , Paupish Lane , Carlow",NaN,INCOMPLETED APPLICATION,PERMISSION,NaN,NaN,NaN,0.0,NaN,0.0,2024-11-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.eplanning.ie/CarlowCC/AppFileRefDet...,No,2026-10-20,NaN,8,Carlow,2024,2024Q4,NaN


In [71]:
# Which counties are in the file?

df_planning['county'].value_counts()

county
Dublin       43351
Cork         24243
Donegal      13785
Galway       13578
Meath        10613
Kildare       9763
Wexford       9604
Tipperary     8926
Wicklow       8259
Kerry         8056
Limerick      7903
Mayo          7530
Clare         7040
Louth         6494
Waterford     6414
Kilkenny      5295
Laois         5238
Offaly        4376
Cavan         4174
Roscommon     4096
Westmeath     3805
Monaghan      3753
Sligo         2916
Carlow        2791
Longford      1772
Leitrim       1366
Name: count, dtype: int64

In [72]:
# Keeping only the Dublin applications and saving them as df_dublin_planning.
# Dublin has 4 councils: Dublin City, Fingal, South Dublin and Dun Laoghaire-Rathdown.

df_dublin_planning = df_planning[df_planning['county'] == 'Dublin'].copy()

print('All of Ireland:', len(df_planning))
print('Dublin only:   ', len(df_dublin_planning))
df_dublin_planning['PlanningAuthority'].value_counts()

All of Ireland: 225141
Dublin only:    43351


PlanningAuthority
Dublin City Council                      18314
Dun Laoghaire Rathdown County Council     9799
Fingal County Council                     8901
South Dublin County Council               6337
Name: count, dtype: int64

In [73]:
# Adding the postal district (same district names as the BER data).
# The planning file has no district column, so we read it from the address:
#   'Crumlin, Dublin 12'                 -> 'Dublin 12'
#   an Eircode 'D16 N2Y3' or 'D16N2Y3'   -> 'Dublin 16'
# We do this now, before the address and postcode columns are removed later.

dublin_districts = ['Dublin 1', 'Dublin 2', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6',
                    'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Dublin 10', 'Dublin 11',
                    'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17',
                    'Dublin 18', 'Dublin 20', 'Dublin 22', 'Dublin 24']

# address and postcode together, in capital letters
address = (df_dublin_planning['DevelopmentAddress'].fillna('') + ' '
           + df_dublin_planning['DevelopmentPostcode'].fillna('')).str.upper()

# 'DUBLIN' followed by a number (maybe with W, for 6W)
district_number = address.str.extract(r'DUBLIN\s*(\d+W?)')[0]

# if there is no 'Dublin 12', look for an Eircode like D12 X2Y3 or D12X2Y3
eircode_number = address.str.extract(r'\bD(\d\d|6W)\s?[0-9A-Z]{4}\b')[0]
district_number = district_number.fillna(eircode_number)

# '04' -> '4' (Eircodes and some councils write a leading zero)
district_number = district_number.str.lstrip('0')

df_dublin_planning['dublin_district'] = 'Dublin ' + district_number

In [74]:
# Anything that is not a real district (no number found, or a typo like 'Dublin 19')
# becomes 'Dublin (no district)' - places like Swords, Lucan or Dun Laoghaire.

not_real = ~df_dublin_planning['dublin_district'].isin(dublin_districts)
df_dublin_planning.loc[not_real, 'dublin_district'] = 'Dublin (no district)'

has_district = (df_dublin_planning['dublin_district'] != 'Dublin (no district)').mean() * 100
print('Applications with a postal district:', round(has_district, 1), '%')
df_dublin_planning['dublin_district'].value_counts()

Applications with a postal district: 70.1 %


dublin_district
Dublin (no district)    12979
Dublin 4                 2054
Dublin 18                1987
Dublin 15                1964
Dublin 14                1964
Dublin 9                 1862
Dublin 6                 1783
Dublin 3                 1734
Dublin 12                1717
Dublin 5                 1644
Dublin 7                 1571
Dublin 8                 1444
Dublin 2                 1422
Dublin 24                1418
Dublin 16                1405
Dublin 13                1314
Dublin 11                1208
Dublin 6W                1069
Dublin 1                  971
Dublin 22                 899
Dublin 17                 356
Dublin 20                 301
Dublin 10                 285
Name: count, dtype: int64

In [75]:
df_dublin_planning.head()

,OBJECTID,PlanningAuthority,ApplicationNumber,DevelopmentDescription,DevelopmentAddress,DevelopmentPostcode,ApplicationStatus,ApplicationType,Decision,LandUseCode,AreaofSite,NumResidentialUnits,OneOffHouse,FloorArea,ReceivedDate,WithdrawnDate,DecisionDate,DecisionDueDate,GrantDate,ExpiryDate,AppealRefNumber,AppealStatus,AppealDecision,AppealDecisionDate,AppealSubmittedDate,FIRequestDate,FIRecDate,LinkAppDetails,OneOffKPI,ETL_DATE,SiteId,word_count,county,year,quarter,days_to_decision,dublin_district
51471,178001,Dublin City Council,3605/23,RETENTION Permission for the following : \r\n(...,"146 Baggot Street Lower, Dublin 2, D02 T990",NaN,Decision Notice Issued,Retention Permission,GRANT RETENTION PERMISSIO,NaN,40.0,NaN,NaN,NaN,2023-04-21,NaN,2023-06-14,2023-06-15,2023-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,36,Dublin,2023,2023Q2,54.0,Dublin 2
51472,178003,Dublin City Council,3631/23,The development will consist of the constructi...,"98-100 South Circular Road, Dublin 8, D08 V5YA",NaN,Decision Notice Issued,Permission,GRANT PERMISSION,NaN,35.0,NaN,NaN,NaN,2023-04-23,NaN,2023-06-19,2023-06-19,2023-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,34,Dublin,2023,2023Q2,57.0,Dublin 8
51473,178004,Dublin City Council,WEB1524/24,Planning permission for attic conversion to us...,"7, Church Park Court, Dublin 6w, D6W DW65",NaN,Decision Notice Issued,Permission,APPLICATION DECLARED INVA,NaN,32.0,NaN,NaN,NaN,2024-04-30,NaN,2024-05-02,2024-06-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,37,Dublin,2024,2024Q2,2.0,Dublin 6W
51474,178005,Dublin City Council,WEB1297/24,The Development will consist of the constructi...,"2 Rathborne Chase, Boulevard, Ashtown, Dublin ...",NaN,Decision Notice Issued,Permission,GRANT PERMISSION,NaN,132.7,NaN,NaN,NaN,2024-03-15,NaN,2024-10-07,2024-10-07,2024-12-03,NaN,NaN,NaN,NaN,NaN,NaN,2024-05-09,2024-09-10,NaN,NaN,2026-07-31,NaN,88,Dublin,2024,2024Q1,206.0,Dublin 15
51475,178006,Dublin City Council,WEB1960/23,The replacement of the existing advertising di...,"151 Pearse Street, Dublin 2, D02 C898",NaN,Decision Notice Issued,Permission,REFUSE PERMISSION,NaN,1.5,NaN,NaN,NaN,2023-11-03,NaN,2024-01-03,2024-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,78,Dublin,2023,2023Q4,61.0,Dublin 2


In [76]:
df_dublin_planning = df_dublin_planning.drop(columns=['OBJECTID'])

In [77]:
df_dublin_planning.isnull().sum()

PlanningAuthority             0
ApplicationNumber             0
DevelopmentDescription        0
DevelopmentAddress            0
DevelopmentPostcode       40879
ApplicationStatus            19
ApplicationType               0
Decision                   1054
LandUseCode               34102
AreaofSite                  194
NumResidentialUnits       32806
OneOffHouse               36020
FloorArea                 29399
ReceivedDate                  0
WithdrawnDate             43075
DecisionDate               1147
DecisionDueDate             323
GrantDate                 16970
ExpiryDate                40304
AppealRefNumber           39908
AppealStatus              42241
AppealDecision            40149
AppealDecisionDate        40153
AppealSubmittedDate       39911
FIRequestDate             35563
FIRecDate                 36271
LinkAppDetails            28113
OneOffKPI                 43351
ETL_DATE                      0
SiteId                    43351
word_count                    0
county  

In [78]:
# Missing values in each column, as a percentage of all Dublin applications

missing_percent = df_dublin_planning.isnull().sum() / len(df_dublin_planning) * 100

missing_percent.round(2).sort_values(ascending=False)

OneOffKPI                 100.00
SiteId                    100.00
WithdrawnDate              99.36
AppealStatus               97.44
DevelopmentPostcode        94.30
ExpiryDate                 92.97
AppealDecisionDate         92.62
AppealDecision             92.61
AppealSubmittedDate        92.06
AppealRefNumber            92.06
FIRecDate                  83.67
OneOffHouse                83.09
FIRequestDate              82.04
LandUseCode                78.66
NumResidentialUnits        75.68
FloorArea                  67.82
LinkAppDetails             64.85
GrantDate                  39.15
DecisionDate                2.65
days_to_decision            2.65
Decision                    2.43
DecisionDueDate             0.75
AreaofSite                  0.45
ApplicationStatus           0.04
DevelopmentAddress          0.00
DevelopmentDescription      0.00
ApplicationNumber           0.00
PlanningAuthority           0.00
ReceivedDate                0.00
ApplicationType             0.00
word_count

In [84]:
# Deleting the columns we don't need. Each group has its reason.
# The question for every column: "would we know this on the day the application arrives?"

# 1. LEAKAGE - these happen AFTER the application arrives, so they give the answer away
#             -> pushed back when further information is asked (55 -> 120 days)
#    FIRequestDate/FIRecDate -> further information, asked during the process
#    DecisionDate         -> days_to_decision is calculated from it
#    GrantDate/ExpiryDate -> only exist if permission was granted
#    WithdrawnDate        -> applicant gave up, no decision
#    ApplicationStatus    -> says the outcome ('Final Grant', 'Appeal decided' ...)
#    Appeal... columns    -> the appeal comes about 27 days after the decision
leakage_columns = ['DecisionDueDate', 'FIRequestDate', 'FIRecDate', 'DecisionDate',
                   'ExpiryDate', 'WithdrawnDate', 'ApplicationStatus','GrantDate',
                   'AppealRefNumber', 'AppealStatus', 'AppealDecision',
                   'AppealSubmittedDate', 'AppealDecisionDate']

# 2. ONLY RECORDED BY SOME COUNCILS - 68% to 83% empty, and the empty values only
#    tell us which council it was (e.g. only Fingal fills NumResidentialUnits)
council_only_columns = ['LandUseCode', 'NumResidentialUnits', 'FloorArea', 'OneOffHouse']

# 3. ADMIN / EMPTY - not about the application
#    ETL_DATE   -> download date (it just repeats the council)
#    LinkAppDetails -> website link
#    OneOffKPI, SiteId -> 100% empty
#    county     -> always 'Dublin'
admin_columns = ['ETL_DATE', 'LinkAppDetails', 'OneOffKPI', 'SiteId', 'county']

# 4. ADDRESS - already used to find dublin_district; the full address is not
#    needed for modelling (data minimisation, as promised in our proposal)
address_columns = ['DevelopmentAddress', 'DevelopmentPostcode']

all_to_delete = leakage_columns + council_only_columns + admin_columns + address_columns

# only delete the ones that are still in the table (so the cell never crashes)
columns_to_delete = []
for column in all_to_delete:
    if column in df_dublin_planning.columns:
        columns_to_delete.append(column)

print('Columns to delete:', len(columns_to_delete))
print('Columns before:', df_dublin_planning.shape[1])
df_dublin_planning = df_dublin_planning.drop(columns=columns_to_delete)
print('Columns after: ', df_dublin_planning.shape[1])
print('Columns kept:', list(df_dublin_planning.columns))

Columns to delete: 1
Columns before: 13
Columns after:  12
Columns kept: ['ApplicationNumber', 'ReceivedDate', 'year', 'quarter', 'dublin_district', 'PlanningAuthority', 'ApplicationType', 'DevelopmentDescription', 'AreaofSite', 'word_count', 'Decision', 'days_to_decision']


In [85]:
# Arranging the columns (nothing is removed - only the order changes):
# id -> dates -> categories -> numbers -> targets

id_columns = ['ApplicationNumber']
date_columns = ['ReceivedDate', 'year', 'quarter']
categorical_columns = ['dublin_district', 'PlanningAuthority', 'ApplicationType', 'DevelopmentDescription']
numeric_columns = ['AreaofSite', 'word_count']          # known only after the decision - never an input
target_columns = ['Decision', 'days_to_decision']

wanted_order = id_columns + date_columns + categorical_columns + numeric_columns 

# 1. columns from our lists, in our order
new_order = []
for column in wanted_order:
    if column in df_dublin_planning.columns:
        new_order.append(column)

# 2. any column we did not list goes here, so it is NOT lost
for column in df_dublin_planning.columns:
    if column not in new_order and column not in target_columns:
        new_order.append(column)
        print('Not in the lists, kept anyway:', column)

# 3. targets always at the end
for column in target_columns:
    if column in df_dublin_planning.columns:
        new_order.append(column)

print('Columns before:', df_dublin_planning.shape[1], ' after:', len(new_order))   # always the same
df_dublin_planning = df_dublin_planning[new_order]

df_dublin_planning.head()

Columns before: 12  after: 12


,ApplicationNumber,ReceivedDate,year,quarter,dublin_district,PlanningAuthority,ApplicationType,DevelopmentDescription,AreaofSite,word_count,Decision,days_to_decision
51471,3605/23,2023-04-21,2023,2023Q2,Dublin 2,Dublin City Council,Retention Permission,RETENTION Permission for the following : \r\n(...,40.0,36,GRANT RETENTION PERMISSIO,54.0
51472,3631/23,2023-04-23,2023,2023Q2,Dublin 8,Dublin City Council,Permission,The development will consist of the constructi...,35.0,34,GRANT PERMISSION,57.0
51473,WEB1524/24,2024-04-30,2024,2024Q2,Dublin 6W,Dublin City Council,Permission,Planning permission for attic conversion to us...,32.0,37,APPLICATION DECLARED INVA,2.0
51474,WEB1297/24,2024-03-15,2024,2024Q1,Dublin 15,Dublin City Council,Permission,The Development will consist of the constructi...,132.7,88,GRANT PERMISSION,206.0
51475,WEB1960/23,2023-11-03,2023,2023Q4,Dublin 2,Dublin City Council,Permission,The replacement of the existing advertising di...,1.5,78,REFUSE PERMISSION,61.0
